## Vector Search with sqlitesearch
Prior approach with minsearch can have some downsides if your dataset is too large. Each time you will need to import data and do the indexing as well converting the documents to vector from scratch each time. Have a persistent database like sqlitesearch helps solve that problem.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../01_module_agentic_rag/.env")

from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("LMSTUDIO_API_KEY"),
    base_url=os.getenv("LMSTUDIO_HOST")
)

In [2]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# Loading data source
from ingest import load_faq_data

documents = load_faq_data()

In [7]:
from tqdm.auto import tqdm

In [9]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [10]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed_model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [11]:
import numpy as np
X = np.array(vectors)
X

array([[-0.02670616, -0.12245759,  0.01594421, ..., -0.0023065 ,
        -0.11218402, -0.02365559],
       [-0.01099559, -0.11074748, -0.02536943, ...,  0.09022236,
        -0.02697359,  0.01975664],
       [-0.08896557, -0.06128179,  0.00775611, ...,  0.04059712,
         0.00479282, -0.02745942],
       ...,
       [-0.03652923,  0.0141543 , -0.06838647, ...,  0.04316789,
         0.08105531, -0.02148631],
       [-0.1309159 , -0.06990598, -0.0093188 , ..., -0.00044342,
        -0.01285736,  0.01426917],
       [-0.07984782,  0.01926984,  0.02544974, ..., -0.03368021,
        -0.01884026,  0.05837055]], shape=(1350, 384), dtype=float32)

In [ ]:
# Initializing sqlitesearch vector database
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [15]:
vs_index.fit(vectors, documents)

In [17]:
query = "I just discovered the course. Can I still join it?"
query_vector = embed_model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [18]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [19]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [20]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [21]:
# Closing the connection
vs_index.close()

In [22]:
# Re-opening the connection when needed

from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
# Another example search
query_vector = embed_model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [25]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

Note: We still load the embedding model to encode the query, but we don't re-embed all the documents. No fit call needed, because the index is already built and waiting on disk.

## Using sqlitesearch vector search in RAG

In [ ]:
# Reopen index
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [26]:
# Import RAG vector class from rag_helper
from rag_helper import RAGBase, LMStudioRAG, RAGVector
model="qwen/qwen3.5-9b"

In [28]:
# Initialize our RAG with persistent vector search
vector_assistant = RAGVector(
    embedder=embed_model,
    index=vs_index, # now using persistent database instead of in-memory database
    llm_client=openai_client,
)

In [30]:
# Running assistant with persistent search database
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up even if the program has already begun. According to the context:\n\n- You can start learning and submit homework **while the form is open**, even without registering. Registration is just used to gauge interest before the course starts.\n- If you want to receive a **certificate**, you need to make sure your project submission happens while they are still accepting submissions for certificates.\n\nSo, signing up late is allowed as long as you act within the submission window for homework and projects—and if you care about the certificate, stay on top of that deadline too.'

In [31]:
# Closing index
vs_index.close()